In [11]:
# 新增法規分塊與檢索
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import re
import json

with open('vision_course_announcements_eng.txt', 'r', encoding='utf-8') as f:
    data_text = f.read()

In [20]:
pattern = r'(?=(公告\s\d+【\d{4}/\d{2}/\d{2}】：|作業[一二三]：))' # for 'vision_course_announcements_eng.txt'
pattern = r'(?=(Course topic of Week \d+\s*\[\d{4}/\d{2}/\d{2}–\d{2}/\d{2}\]:|Announcement\s\d+\s*\[\d{4}/\d{2}/\d{2}\]:|Assignment[123]:))' # for 'vision_course_announcements_eng.txt'
# 先用 re.split 拆分，會保留分隔符作為元素
chunks = re.split(pattern, data_text)

data_chunks = []
for i in range(1, len(chunks), 2):
    data_chunks.append(chunks[i+1])

# 輸出每段分割結果
for segment in data_chunks:
    print("------段落------")
    print(segment)

------段落------
Course topic of Week 1 [2025/02/17–02/21]:Course introduction, basic image processing concepts, reading and displaying grayscale images

------段落------
Course topic of Week 2 [2025/02/24–02/28]:Basic operations in NumPy and OpenCV (matrix operations, image I/O, drawing)

------段落------
Course topic of Week 3 [2025/03/03–03/07]:Convolution operations and edge detection (Sobel, Prewitt)

------段落------
Course topic of Week 4 [2025/03/10–03/14]:Smoothing and blurring filters (mean filter, Gaussian filter)

------段落------
Course topic of Week 5 [2025/03/17–03/21]:Basics of Convolutional Neural Networks (CNN) and their application in image classification

------段落------
Course topic of Week 6 [2025/03/24–03/28]:Binarization and thresholding (including Otsu's method)

------段落------
Course topic of Week 7 [2025/03/31–04/04]:Local operations: morphological operations (dilation, erosion, opening/closing)

------段落------
Course topic of Week 8 [2025/04/07–04/11]:Geometric transfo

In [23]:
# 建立嵌入模型
embedder = SentenceTransformer('all-MiniLM-L6-v2')
# 分批處理, 將 law_chunks 分成小批次生成嵌入向量，避免一次性處理全部數據
batch_size = 32  # 根據 GPU 記憶體調整（從 32 開始嘗試）
chunk_embeddings = []

for i in range(0, len(data_chunks), batch_size):
    batch = data_chunks[i:i+batch_size]
    embeddings = embedder.encode(batch, convert_to_numpy=True, show_progress_bar=True)
    chunk_embeddings.append(embeddings)

chunk_embeddings = np.concatenate(chunk_embeddings, axis=0)
# chunk_embeddings = embedder.encode(law_chunks, convert_to_numpy=True, show_progress_bar=True)

# 建立FAISS索引
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


In [24]:
def retrieve_law_context(question, top_k=10):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    D, I = index.search(q_emb, top_k)
    return [data_chunks[idx] for idx in I[0]]

In [26]:
import time
from datetime import datetime, timedelta, timezone

def get_taiwan_time():
    """取得台灣時間 (GMT+8)"""
    # 創建台灣時區 (UTC+8)
    tw_timezone = timezone(timedelta(hours=8))
    # 獲取目前UTC時間並轉換為台灣時間
    tw_time = datetime.now(tw_timezone)
    # 格式化時間字串
    print(f"tw_time:{tw_time}")
    print(f"現在時間:{datetime.now()}")
    return tw_time.strftime("%Y-%m-%d %H:%M:%S")

In [29]:
q = f"what is the course topic of last week"#, TODAY’S DATE: {get_taiwan_time()}"
context = "\n\n".join(retrieve_law_context(q, top_k=5))
print(context)

Course topic of Week 18	[2025/06/16–06/20]:Final exam (held on 2025/06/16, covering Weeks 10–16)



Course topic of Week 17	[2025/06/09–06/13]:Final project presentations and summary preparation


Course topic of Week 9 [2025/04/14–04/18]:Midterm exam (held on 2025/04/14, covering Weeks 1–8)


Course topic of Week 15	[2025/05/26–05/30]:Project mentoring week (group discussions and progress checks)


Announcement 6 [2025/03/25]:Course videos have been uploaded to the teaching platform. Please watch them and complete the video quiz this week.

